[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/06_Relation_Extraction.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 关系抽取（Relation Extraction）—— 全面指南

## 概览

本 notebook 提供一份**全面指南**，讲解如何用 Semantica 的关系抽取模块抽取实体之间的关系并构建 RDF 三元组，为构建知识图谱准备数据。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/semantic_extract/)

### 先看主线

本篇的八个步骤不是八种并列的功能演示，而是一条完整的加工链，分两段：

**理解层（第 1–4 步）—— 文本说了什么**

1. 第 1 步 基础抽取：在实体之间"画线"，确立关系的最小概念
2. 第 2 步 方法对比：线有好几种画法，各有取舍
3. 第 3 步 高级配置：画法选定后，给画线立规矩
4. 第 4 步 关系分类：画好的线按标签分堆，为建图铺路

**入库层（第 5–7 步）—— 图谱里存什么**

5. 第 5 步 三元组抽取：每条线定型成标准"档案卡"
6. 第 6 步 RDF 序列化：档案卡写成通用交换格式
7. 第 7 步 完整流水线：整条链串起来跑一遍

第 8 步收尾，总结生产环境的最佳实践。

前四步面向**理解文本**：抽出的关系是松散的、带着文本证据的中间产物，给人看、给调试看。后三步面向**构建图谱**：三元组是严格定型的存储单元，给机器用。两段的衔接点在第 5 步——已经圈好的实体、画好的线，正好作为三元组抽取的锚点，先粗后精。

### 学习目标

完成本 notebook 后，你将能够：

- 使用 `RelationExtractor` 抽取关系
- 理解不同的抽取方法（pattern、dependency、co-occurrence、HuggingFace、LLM）
- 配置抽取参数以获得最佳效果
- 用 `TripletExtractor` 抽取 RDF 三元组
- 用 `RDFSerializer` 把三元组序列化为 RDF 格式
- 构建完整的 实体 → 关系 → 三元组 流水线

### 你将学到什么

| 组件 | 用途 | 适用场景 |
|-----------|---------|-------------|
| `RelationExtractor` | 抽取实体关系 | 发现关联 |
| `TripletExtractor` | 抽取 RDF 三元组 | 构建知识图谱 |
| `RDFSerializer` | 序列化为 RDF 格式 | 数据导出 |

---

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

---

In [ ]:
!pip install semantica


## 第 1 步：基础关系抽取

主线进入理解层。第一件事：在两个实体之间"画一条带标签的线"。上一篇已经圈出了实体（人名、公司名），这一步判断谁和谁之间有线、线上的标签是什么。工具是 `RelationExtractor`。

### 什么是 RelationExtractor？

`RelationExtractor` 识别实体之间的关系：
- 发现 "founded_by"、"located_in"、"works_for" 这类关联
- 返回包含主语（subject）、谓语（predicate）、宾语（object）的 Relation 对象
- 支持多种抽取方法
- 为每条关系提供置信度分数

### 理解关系

一条关系由三部分组成——正好对应一条线的两端和标签：
- **主语（Subject）**：源实体，线的一端（如 "Apple Inc."）
- **谓语（Predicate）**：关系类型，线上的标签（如 "founded_by"）
- **宾语（Object）**：目标实体，线的另一端（如 "Steve Jobs"）

In [ ]:
from semantica.semantic_extract import RelationExtractor, NERExtractor

# Initialize extractors
ner_extractor = NERExtractor()
relation_extractor = RelationExtractor()

# Sample text with clear relationships
text = """
Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 1976.
The company is headquartered in Cupertino, California. Tim Cook is the current CEO
of Apple Inc. and took over from Steve Jobs in August 2011.
"""

# First, extract entities
entities = ner_extractor.extract(text)
print(f" Extracted {len(entities)} entities\n")

# Then, extract relationships
relationships = relation_extractor.extract(text, entities)

print(f" Extracted {len(relationships)} relationships:\n")
print("=" * 80)

for i, rel in enumerate(relationships, 1):
    # Handle both dict and object formats
    source = rel.get('source', rel.get('subject', '')) if isinstance(rel, dict) else getattr(rel, 'subject', '')
    target = rel.get('target', rel.get('object', '')) if isinstance(rel, dict) else getattr(rel, 'object', '')
    rel_type = rel.get('type', rel.get('predicate', 'related_to')) if isinstance(rel, dict) else getattr(rel, 'predicate', 'related_to')
    confidence = rel.get('confidence', 1.0) if isinstance(rel, dict) else getattr(rel, 'confidence', 1.0)
    
    # Get source and target text
    if isinstance(source, dict):
        source_text = source.get('text', source.get('entity', str(source)))
    else:
        source_text = getattr(source, 'text', str(source))
    
    if isinstance(target, dict):
        target_text = target.get('text', target.get('entity', str(target)))
    else:
        target_text = getattr(target, 'text', str(target))
    
    print(f"{i:2d}. {source_text:20s} --[{rel_type:15s}]--> {target_text:20s} (conf: {confidence:.2f})")

print("=" * 80)

### 理解 Relation 对象

画出来的每条线长什么样？每条抽取出的关系包含：

| 属性 | 描述 | 示例 |
|-----------|-------------|----------|
| `subject` | 源实体 | Entity("Apple Inc.") |
| `predicate` | 关系类型 | "founded_by" |
| `object` | 目标实体 | Entity("Steve Jobs") |
| `confidence` | 抽取置信度（0-1） | 0.85 |
| `context` | 周边文本 | "Apple Inc. was founded by Steve Jobs" |
| `metadata` | 附加信息 | {"method": "pattern"} |

### 常见关系类型

标签库里常见这几类：

- **founded_by**：组织由某人创立
- **located_in**：实体位于某地
- **works_for**：人为某组织工作
- **born_in**：人出生于某地
- **part_of**：实体是另一实体的一部分
- **related_to**：通用关系

## 第 2 步：不同的抽取方法

线有好几种画法。Semantica 支持多种关系抽取方法，各有取舍：

### 方法对比

| 方法 | 速度 | 准确率 | 适用场景 | 依赖 |
|--------|-------|----------|----------|----------|
| **pattern** |  | ⭐⭐⭐ | 常见关系 | 无 |
| **dependency** |  | ⭐⭐⭐⭐ | 语法关系 | spaCy |
| **cooccurrence** |  | ⭐⭐ | 基于邻近共现 | 无 |
| **huggingface** |  | ⭐⭐⭐⭐⭐ | 领域特定 | HF 模型 |
| **llm** |  | ⭐⭐⭐⭐⭐ | 复杂、自定义 | API key |

In [ ]:
from semantica.semantic_extract import RelationExtractor

sample_text = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
sample_entities = ner_extractor.extract(sample_text)

print(" Comparing Relation Extraction Methods:\n")
print("=" * 80)

# Try different methods
methods_to_try = ["pattern", "dependency", "cooccurrence"]

for method_name in methods_to_try:
    try:
        print(f"\n Method: {method_name.upper()}")
        print("-" * 40)
        
        extractor = RelationExtractor(method=method_name)
        relations = extractor.extract(sample_text, sample_entities)
        
        print(f"Found {len(relations)} relations:")
        for rel in relations[:3]:  # Show first 3
            source = rel.get('source', rel.get('subject', '')) if isinstance(rel, dict) else getattr(rel, 'subject', '')
            target = rel.get('target', rel.get('object', '')) if isinstance(rel, dict) else getattr(rel, 'object', '')
            rel_type = rel.get('type', rel.get('predicate', 'related_to')) if isinstance(rel, dict) else getattr(rel, 'predicate', 'related_to')
            
            # Get text representations
            source_text = source.get('text', str(source)) if isinstance(source, dict) else getattr(source, 'text', str(source))
            target_text = target.get('text', str(target)) if isinstance(target, dict) else getattr(target, 'text', str(target))
            
            print(f"  • {source_text} --[{rel_type}]--> {target_text}")
            
    except Exception as e:
        print(f"  ️  Method '{method_name}' not available: {str(e)[:50]}")

print("\n" + "=" * 80)

## 第 3 步：通过配置做高级关系抽取

画法选定之后，还要给画线立规矩：只画哪几种线、把握不够的不画、隔太远的不连。这不是新的抽取能力，仍是第 1 步那个 `RelationExtractor`，只是调参控质量：

### 关键参数：

- **`relation_types`**：只抽取这些关系类型（如 `["founded", "works_at"]`）
- **`bidirectional`**：是否抽取双向关系（默认 False）
- **`confidence_threshold`**：最低置信度分数（0.0-1.0，默认 0.6）
- **`max_distance`**：实体之间的最大 token 距离（默认 50）

In [ ]:
# Create extractor with custom configuration
advanced_extractor = RelationExtractor(
    relation_types=["founded_by", "located_in", "works_for"],  # Only extract these types
    confidence_threshold=0.7,  # Higher threshold for quality
    bidirectional=False,  # One-way relations only
    max_distance=50  # Max 50 tokens between entities
)

# Sample texts
texts = [
    "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico.",
    "Satya Nadella works for Microsoft as the CEO.",
    "Google is located in Mountain View, California.",
    "Amazon was founded by Jeff Bezos in Seattle, Washington."
]

print(" Advanced Relation Extraction:\n")
print("=" * 80)

for i, text in enumerate(texts, 1):
    entities = ner_extractor.extract(text)
    relations = advanced_extractor.extract(text, entities)
    
    print(f"\n Text {i}: {text}")
    print(f"   Relations found: {len(relations)}")
    
    for rel in relations:
        source = rel.get('source', rel.get('subject', '')) if isinstance(rel, dict) else getattr(rel, 'subject', '')
        target = rel.get('target', rel.get('object', '')) if isinstance(rel, dict) else getattr(rel, 'object', '')
        rel_type = rel.get('type', rel.get('predicate', 'related_to')) if isinstance(rel, dict) else getattr(rel, 'predicate', 'related_to')
        confidence = rel.get('confidence', 1.0) if isinstance(rel, dict) else getattr(rel, 'confidence', 1.0)
        
        source_text = source.get('text', str(source)) if isinstance(source, dict) else getattr(source, 'text', str(source))
        target_text = target.get('text', str(target)) if isinstance(target, dict) else getattr(target, 'text', str(target))
        
        print(f"     • {source_text} --[{rel_type}]--> {target_text} (conf: {confidence:.2f})")

print("\n" + "=" * 80)

## 第 4 步：关系分类

线画完了，摊在桌上还只是一堆结果。这一步按谓语类型把关系分堆整理。注意：整理本身不产生新信息，但为后面建图铺路：

- **喂给图谱构建**：同一类关系对应图谱上同一种边类型，分好组就能按组批量映射
- **排查质量**：哪类关系抽得少、哪类全是低置信度，分完堆一眼可见
- **统一类型**：和上一篇 `EntityClassifier` 归一实体类型是同一件事的关系版，谓语统一了，三元组才不会一个意思三种写法

In [ ]:
# Extract relations from all texts
all_relations = []
for text in texts:
    entities = ner_extractor.extract(text)
    relations = advanced_extractor.extract(text, entities)
    all_relations.extend(relations)

# Classify relations
classified_relations = advanced_extractor.classify_relations(all_relations)

print("️  Relation Classification:\n")
print("=" * 80)

for rel_type, rel_list in sorted(classified_relations.items()):
    print(f"\n{rel_type.upper()} ({len(rel_list)} relations):")
    print("-" * 40)
    
    for rel in rel_list:
        source = rel.get('source', rel.get('subject', '')) if isinstance(rel, dict) else getattr(rel, 'subject', '')
        target = rel.get('target', rel.get('object', '')) if isinstance(rel, dict) else getattr(rel, 'object', '')
        
        source_text = source.get('text', str(source)) if isinstance(source, dict) else getattr(source, 'text', str(source))
        target_text = target.get('text', str(target)) if isinstance(target, dict) else getattr(target, 'text', str(target))
        
        print(f"  • {source_text} → {target_text}")

print("\n" + "=" * 80)

## 第 5 步：三元组抽取

主线在这里从理解层跨入入库层。关系是画在文本上的线，松散、带证据，适合分析；要让图谱存得住、查得动，每条线要定型成一张格式统一的"档案卡"——三元组 (主语, 谓语, 宾语)。工具是 `TripletExtractor`。

先澄清一个常见误解：三元组抽取并不依赖前面几步才能跑，直接喂文本就能出结果（本步示例就是这么写的）。那前面为什么还要先抽关系？因为已经圈好的实体、画好的线是现成的锚点——第 7 步的流水线会把 entities 和 relations 一起传进来，三元组抽取只需定型、补上时间和溯源，不必从零重读文本。先粗后精，而不是"必须先 A 才能 B"。

### 什么是 RDF 三元组？

RDF（资源描述框架，Resource Description Framework）三元组是包含三部分的陈述：
- **主语（Subject）**：谈论的对象
- **谓语（Predicate）**：属性或关系
- **宾语（Object）**：值或目标

示例：`(Apple Inc., founded_by, Steve Jobs)`

### 为什么用三元组？

- 知识表示的**标准格式**
- 与 RDF 数据库和语义网**兼容**
- 可用 SPARQL **查询**
- 跨系统**互操作**

In [ ]:
from semantica.semantic_extract import TripletExtractor

# Initialize triplet extractor
triplet_extractor = TripletExtractor(
    include_temporal=True,  # Include temporal information
    include_provenance=True  # Track source sentences
)

# Sample text
triplet_text = """
Apple Inc. was founded by Steve Jobs in 1976. The company is based in Cupertino, California.
Tim Cook became CEO in 2011. Apple develops the iPhone and MacBook products.
"""

# Extract triplets
triplets = triplet_extractor.extract_triplets(triplet_text)

print(f" Extracted {len(triplets)} RDF Triplets:\n")
print("=" * 80)

for i, triplet in enumerate(triplets, 1):
    subject = triplet.get('subject', '') if isinstance(triplet, dict) else triplet.subject
    predicate = triplet.get('predicate', '') if isinstance(triplet, dict) else triplet.predicate
    obj = triplet.get('object', '') if isinstance(triplet, dict) else triplet.object
    confidence = triplet.get('confidence', 1.0) if isinstance(triplet, dict) else getattr(triplet, 'confidence', 1.0)
    
    print(f"{i:2d}. ({subject}, {predicate}, {obj})")
    print(f"    Confidence: {confidence:.2f}")
    
    # Show temporal info if available
    metadata = triplet.get('metadata', {}) if isinstance(triplet, dict) else getattr(triplet, 'metadata', {})
    if metadata.get('temporal'):
        print(f"    Temporal: {metadata['temporal']}")
    print()

print("=" * 80)

## 第 6 步：RDF 序列化

档案卡定了内容，还要选一种通用的书写格式，才能交给 RDF 数据库和语义网生态。`RDFSerializer` 把三元组序列化成多种 RDF 格式：

### 支持的格式：

| 格式 | 扩展名 | 适用场景 |
|--------|-----------|----------|
| **Turtle** | .ttl | 人类可读、紧凑 |
| **N-Triples** | .nt | 简单、按行组织 |
| **JSON-LD** | .jsonld | Web 友好、基于 JSON |
| **RDF/XML** | .rdf | 基于 XML、较冗长 |

In [ ]:
from semantica.semantic_extract import RDFSerializer

# Initialize serializer
serializer = RDFSerializer()

print(" RDF Serialization Examples:\n")
print("=" * 80)

# Serialize to different formats
formats = ["turtle", "ntriples", "jsonld"]

for fmt in formats:
    print(f"\n {fmt.upper()} Format:")
    print("-" * 40)
    
    try:
        serialized = serializer.serialize_to_rdf(triplets[:3], format=fmt)  # Show first 3
        
        # Show preview (first 300 chars)
        preview = serialized[:300] + "..." if len(serialized) > 300 else serialized
        print(preview)
        
    except Exception as e:
        print(f"Error: {str(e)[:50]}")

print("\n" + "=" * 80)

## 第 7 步：完整抽取流水线

前面各步单独看过一遍，现在串成整条链：**实体 → 关系 → 三元组**。留意 `extract_triplets` 的调用方式——实体和关系作为锚点传入，正是第 5 步说的"先粗后精"。这条链就是构建知识图谱的完整工作流。

In [ ]:
def extract_knowledge(text):
    """
    Complete knowledge extraction pipeline.
    
    Args:
        text: Input text
        
    Returns:
        dict: Extracted entities, relations, and triplets
    """
    # Step 1: Extract entities
    entities = ner_extractor.extract(text)
    
    # Step 2: Extract relations
    relations = relation_extractor.extract(text, entities)
    
    # Step 3: Extract triplets
    triplets = triplet_extractor.extract_triplets(text, entities=entities, relationships=relations)
    
    return {
        'entities': entities,
        'relations': relations,
        'triplets': triplets
    }

# Sample knowledge-rich text
knowledge_text = """
Tesla Inc. was founded by Elon Musk, JB Straubel, Martin Eberhard, Marc Tarpenning, 
and Ian Wright in 2003. The company is headquartered in Austin, Texas. Tesla produces 
electric vehicles including the Model S, Model 3, Model X, and Model Y. Elon Musk serves 
as CEO and has been instrumental in the company's growth.
"""

print(" Complete Extraction Pipeline:\n")
print("=" * 80)

# Run pipeline
result = extract_knowledge(knowledge_text)

print(f"\n Extraction Results:")
print("-" * 40)
print(f"Entities extracted: {len(result['entities'])}")
print(f"Relations extracted: {len(result['relations'])}")
print(f"Triplets extracted: {len(result['triplets'])}")

print(f"\n Sample Triplets:")
for i, triplet in enumerate(result['triplets'][:5], 1):
    subject = triplet.get('subject', '') if isinstance(triplet, dict) else triplet.subject
    predicate = triplet.get('predicate', '') if isinstance(triplet, dict) else triplet.predicate
    obj = triplet.get('object', '') if isinstance(triplet, dict) else triplet.object
    print(f"  {i}. ({subject}, {predicate}, {obj})")

print("\n" + "=" * 80)

## 第 8 步：最佳实践与技巧

链条走完，收尾讲生产环境的经验：

### 选择合适的方法

1. 常见关系**从 pattern 开始**
2. 要语法准确性**用依存分析（dependency parsing）**
3. 探索性分析**试试共现（co-occurrence）**
4. 复杂、领域特定的关系**考虑 LLM**

### 优化抽取

- **设置置信度阈值**（生产环境 0.6-0.7）
- **指定 relation_types** 聚焦抽取
- 按文本结构**调整 max_distance**

### 常见的坑

- **不要**跳过实体抽取（关系依赖实体）
- **不要**用很低的置信度阈值
- **不要**忘记序列化三元组以便存储

### 各组件的适用场景

| 使用场景 | 推荐组件 |
|----------|----------------------|
| 发现实体关联 | `RelationExtractor` |
| 构建知识图谱 | `TripletExtractor` |
| 导出为 RDF | `RDFSerializer` |

### 性能技巧

1. **实体只抽一次**，供关系和三元组复用
2. 多文档**批处理**
3. **复用抽取器**，不要反复重建
4. **尽早用置信度阈值过滤**

## 小结

### 回顾主线

本篇八步是一条加工链，分两段：

- **理解层（第 1–4 步）**回答"文本说了什么"：画线（基础抽取、方法选择）、立规矩（配置）、分堆（分类），产物是松散带证据的关系。
- **入库层（第 5–7 步）**回答"图谱里存什么"：定型（三元组）、书写（RDF 序列化）、串联（流水线），产物是严格标准化的存储单元。

衔接点在第 5 步：理解层的结果作为锚点传给三元组抽取，先粗后精。

### 你学到了什么

在本 notebook 中，你学会了：

用 `RelationExtractor` **抽取关系**  
**对比抽取方法**（pattern、dependency、co-occurrence、HuggingFace、LLM）  
**配置抽取参数**以获得最佳效果  
用 `TripletExtractor` **抽取 RDF 三元组**  
用 `RDFSerializer` **序列化为 RDF 格式**  
构建从实体到三元组的**完整流水线**  

### 核心要点

1. **关系连接实体**：构成知识图谱的骨架
2. **多种方法可选**：按准确率与速度权衡
3. **配置很强大**：为你的领域调好参数
4. **三元组是标准**：用 RDF 保证互操作性
5. **流水线高效**：按 实体 → 关系 → 三元组 顺序抽取

### 下一步

**下一篇**：[07_Building_Knowledge_Graphs.ipynb](./07_Building_Knowledge_Graphs.ipynb)  
学习如何用抽取出的三元组构建完整知识图谱！

**延伸阅读**：
- [Semantic Extract API 参考](https://semantica.readthedocs.io/reference/semantic_extract/)
- [知识图谱模块](https://semantica.readthedocs.io/reference/kg/)
- [高级图分析](../advanced/02_Advanced_Graph_Analytics.ipynb)

---

**有疑问或问题？** 欢迎访问我们的 [GitHub 仓库](https://github.com/semantica-agi/semantica)或[文档](https://semantica.readthedocs.io)。